# Lab 1 — Gemini API basics

Notes to self while exploring the `google-genai` SDK: interactions, streaming, multi-turn context, and structured (JSON) output with Pydantic.

Needs a `.env` with `GEMINI_API_KEY=...`.

---
## 1. Load the API key

Keep the key in `.env` (git-ignored) and load it into env vars rather than hard-coding it.

In [16]:
# Step 1: Load the API keys into environment variables from the .env file
from dotenv import load_dotenv

print("Loading API keys from .env file...")

# Set override=True to ensure that the .env file values take precedence over existing environment variables
load_dotenv(override=True) 

Loading API keys from .env file...


True

In [17]:
# Step 2: Access the Gemini API key from the environment variables
import os

gemini_api_key = os.getenv("GEMINI_API_KEY")

if gemini_api_key:
    print(f"Gemini API key loaded successfully and begins {gemini_api_key[:4]}")
else:
    print("Gemini API key not found. Please check your .env file.")

Gemini API key loaded successfully and begins AQ.A


In [18]:
# Step 3: Import libraries
from google import genai

print("Libraries imported successfully.")
print("Starting the Gemini API interaction...")
print("="*60)

Libraries imported successfully.
Starting the Gemini API interaction...


---
## 2. Initialize the client

`genai.Client()` reads `GEMINI_API_KEY` from the env automatically. Pin the model name in `MODEL` so it's easy to swap.

In [19]:
# Initialize the client
client = genai.Client()
MODEL = "gemini-3.5-flash"

---
## 3. First interaction

One request/response = an *interaction*. 
The API returns the full Interaction resource containing metadata, usage statistics, and the step-by-step history of the turn.
While the SDKs expose the full response, they also provide convenience properties like interaction.output_text and interaction.output_image to access final outputs directly.

https://ai.google.dev/gemini-api/docs/interactions-overview

In [ ]:
# Create an interaction with the Gemini API
interaction = client.interactions.create(
    model=MODEL,
    input="Explain how AI works in few sentences"
)
print(interaction)




status='completed' model='gemini-3.5-flash' agent=None id='v1_Chd3TXhzYXRMVE1kSEgxZThQMkpic3VBcxIXd014c2F0TFRNZEhIMWU4UDJKYnN1QXM' created='2026-07-31T16:26:40Z' updated='2026-07-31T16:26:40Z' system_instruction=None tools=None usage=Usage(cached_tokens_by_modality=None, grounding_tool_count=None, input_tokens_by_modality=[ModalityTokens(modality='text', tokens=8)], output_tokens_by_modality=None, tool_use_tokens_by_modality=None, total_cached_tokens=0, total_input_tokens=8, total_output_tokens=67, total_thought_tokens=535, total_tokens=610, total_tool_use_tokens=0) response_modalities=None response_mime_type=None previous_interaction_id=None environment_id=None service_tier='standard' webhook_config=None steps=[ThoughtStep(signature='EqQUCqEUARFNMg+ASZrBr+Vt+MDjn6LSK5xaxYtid5eixNCxajYdzL405InVVYNPzvib73VaF3qqrAnHjuCSG05jZeeAdOTwisHiFNYBZllXQm1IqQt5aMPSRcrvR/7tjCEgGn6rYCi0O5A32PHBFM9sTwZgmX/fIeKew/XGB+Yb+vaNsXkSKgWn+vBdSeh9sjjIZNHPZzA/OdS39SuWyZknPAB2Me6ACeIl0HYGkQQkn4k8QElU/jmDtEetd3g

In [21]:
interaction.output_text

'Artificial Intelligence (AI) works by training computer systems on massive amounts of data using mathematical algorithms. By analyzing this information, the AI learns to recognize underlying patterns and relationships, similar to how humans learn from experience. Once trained, it uses these learned patterns to make predictions, solve complex problems, and generate new content when given new instructions.'

---
## 4. Streaming

`stream=True`

In [22]:
# Stream the response
stream = client.interactions.create(
    model=MODEL,
    input="Explain how AI works in few sentences",
    stream=True
)

for event in stream:
    print(event)

interaction=InteractionSseEventInteraction(id='v1_Chd3OHhzYXZiTkdyeXp2cjBQaUtDSXVBcxIXdzh4c2F2Yk5Hcnl6dnIwUGlLQ0l1QXM', status='in_progress', object='interaction', model='gemini-3.5-flash', agent=None, created=None, updated=None, service_tier=None, usage=None, steps=None) event_type='interaction.created' event_id=None metadata=None
interaction_id='v1_Chd3OHhzYXZiTkdyeXp2cjBQaUtDSXVBcxIXdzh4c2F2Yk5Hcnl6dnIwUGlLQ0l1QXM' status='in_progress' event_id=None event_type='interaction.status_update' metadata=None
index=0 step=ThoughtStep(signature=None, summary=None, type='thought') event_id=None event_type='step.start' metadata=None
delta=ThoughtSignatureDelta(signature='Er8UCrwUARFNMg8PjM+Sv/7YeJYG3W+jKjeZQzLaUL/nqiwe/leEmtKKJKJFuuE3Fv+i7b3HnY9gXnpq/DsFB9bg1va4Jh48hqbOl1VP0IULboJTulPeFXs5m0GhVS0fPsDaQ24LBPUVLA8ObHxWIKnn/otXPg46s2oJGNTB3ZzDOtYBJFKf7Y0zTIeiycqZQb8rHycT4kcEMSgaxkh9yMbyAYSt5GikkCQFPQ/ScTic1MsBMBGzppLO/mwWUbVa/8QGufYGWs/LmmXitECJ5npB9XHXakG0xnrDfV51cyhbijPn6qjSE+EbuGAYNuN6pOFokeRc

---
## 5. Multi-turn
Continue a conversation on the server using `previous_interaction_id`

In [23]:
# Multi-turn conversation
# See more in https://ai.google.dev/gemini-api/docs/text-generation#multi-turn-conversations
interaction1 = client.interactions.create(
    model=MODEL,
    input="I have two dogs in my house."
)
print(f"Response 1: {interaction1.output_text}")

interaction2 = client.interactions.create(
    model=MODEL,
    input="How many paws are in my house?",
    previous_interaction_id=interaction1.id
)
print(f"Response 2: {interaction2.output_text}")

Response 1: That sounds like double the fun (and probably double the mischief)! 

What are their names and what breeds are they? I'd love to hear about their personalities!
Response 2: Since you have two dogs, there are **8 paws** in your house! 

(Plus your own 2 feet, of course!) Of course, if you have any other pets hiding around, that number might go up!


In [ ]:
# Multi-turn conversation with a streamed follow-up
interaction1 = client.interactions.create(
    model=MODEL,
    input="I have 2 dogs in my house.",
)
print(interaction1.output_text)

# Link this turn to the previous one so the model keeps the conversation context
stream = client.interactions.create(
    model=MODEL,
    input="How many paws are in my house?",
    previous_interaction_id=interaction1.id,
    stream=True
)
# Keep only the text deltas and print them incrementally (skip lifecycle/thought events)
for event in stream:
    if event.event_type == "step.delta":
        if event.delta.type == "text":
            print(event.delta.text, end="")

<h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try this commercial application:<br/>
            First ask the LLM to pick a business area that might be worth exploring for an Agentic AI opportunity.<br/>
            Then ask the LLM to present a pain-point in that industry - something challenging that might be ripe for an Agentic solution.<br/>
            Finally have 3 third LLM call propose the Agentic AI solution. <br/>
            We will cover this at up-coming labs, so don't worry if you're unsure.. just give it a try!
            </span>

---
## 6. Structured output (Pydantic)

In [57]:
from pydantic import BaseModel, Field
class BusinessArea(BaseModel):
    business_area: str = Field(description="A business area that might be worth exploring for an Agentic AI opportunity.")
    description: str = Field(description="A brief description of this business area.")
    reasoning: str = Field(description="The reasoning behind why this business area is worth exploring for an Agentic AI opportunity.")

interaction1 = client.interactions.create(
    model=MODEL,
    input=(
        "Pick a single, high-value business area or industry that is "
        "worth exploring for an Agentic AI opportunity. Name the industry "
        "and explain in 2-3 sentences why it is a strong candidate."
    ),
    response_format= {
        "type": "text",
        "mime_type": "application/json",
        "schema": BusinessArea.model_json_schema()
    },
)

interaction1.output_text

'{\n  "business_area": "Supply Chain Exception Management",\n  "description": "The autonomous monitoring, rerouting, and negotiation of shipments and inventory levels in response to real-world disruptions like weather, port delays, or sudden supplier shortages.",\n  "reasoning": "This area is a prime candidate for Agentic AI because resolving supply chain disruptions currently requires labor-intensive coordination across fragmented third-party legacy systems and real-time communication. Autonomous agents can proactively analyze alternative routes, negotiate pricing with backup carriers, and execute contract modifications in seconds rather than days. By shifting from reactive human intervention to autonomous, self-healing logistics, enterprises can drastically reduce costly downtime and minimize inventory holding costs."\n}'

In [58]:
business_area = BusinessArea.model_validate_json(interaction1.output_text)
business_area


BusinessArea(business_area='Supply Chain Exception Management', description='The autonomous monitoring, rerouting, and negotiation of shipments and inventory levels in response to real-world disruptions like weather, port delays, or sudden supplier shortages.', reasoning='This area is a prime candidate for Agentic AI because resolving supply chain disruptions currently requires labor-intensive coordination across fragmented third-party legacy systems and real-time communication. Autonomous agents can proactively analyze alternative routes, negotiate pricing with backup carriers, and execute contract modifications in seconds rather than days. By shifting from reactive human intervention to autonomous, self-healing logistics, enterprises can drastically reduce costly downtime and minimize inventory holding costs.')

In [59]:
class PainPoint(BaseModel):
    pain_point: str = Field(description="A pain point for the business area.")
    description: str = Field(description="A brief description of why this pain point.")

interaction2 = client.interactions.create(
    model=MODEL,
    input="In the business area selected, present a single, high-value pain point that an Agentic AI could address. Explain in 2-3 sentences why this pain point is significant.",
    previous_interaction_id=interaction1.id,
    response_format= {
        "type": "text",
        "mime_type": "application/json",
        "schema": PainPoint.model_json_schema()
    },
)
interaction2.output_text

'{\n  "pain_point": "Manual, multi-party coordination and negotiation latency during shipment disruptions",\n  "description": "When unexpected transit disruptions occur, logistics teams must manually coordinate across disjointed emails, phone calls, and carrier portals to find alternative routes and negotiate rates. This highly manual process takes hours or days to resolve, resulting in expensive expedited shipping fees, stockouts, and missed customer SLAs. Automating this multi-party negotiation through Agentic AI can compress resolution times from days to minutes, directly protecting profit margins and supply chain reliability."\n}'

In [60]:
pain_point = PainPoint.model_validate_json(interaction2.output_text)
pain_point

PainPoint(pain_point='Manual, multi-party coordination and negotiation latency during shipment disruptions', description='When unexpected transit disruptions occur, logistics teams must manually coordinate across disjointed emails, phone calls, and carrier portals to find alternative routes and negotiate rates. This highly manual process takes hours or days to resolve, resulting in expensive expedited shipping fees, stockouts, and missed customer SLAs. Automating this multi-party negotiation through Agentic AI can compress resolution times from days to minutes, directly protecting profit margins and supply chain reliability.')

In [61]:
class Solution(BaseModel):
    solution: str = Field(description="A solution for the pain point.")
    description: str = Field(description="A brief description of this solution")
    reasoning: str = Field(description="A brief description of the reasoning behind this solution.")
    
interaction3 = client.interactions.create(
    model=MODEL,
    input="Propose an Agentic AI solution for this pain point for the selected business area. Explain in 2-3 sentences why this solution is effective.",
    previous_interaction_id=interaction2.id,
    response_format= {
        "type": "text",
        "mime_type": "application/json",
        "schema": Solution.model_json_schema()
    },
)
interaction3.output_text

'{\n  "solution": "Autonomous Logistics Exception Resolver (ALER)",\n  "description": "An Agentic AI system that continuously monitors freight telemetry to detect transit delays, automatically calculates optimal rerouting options, and autonomously negotiates and books backup capacity with third-party carriers.",\n  "reasoning": "This solution is highly effective because it bridges the gap between passive alerting and active resolution by executing complex, multi-party workflows across disparate systems like APIs, EDI, and conversational email. By autonomously negotiating spot rates and confirming alternative bookings in real-time, the agent resolves disruptions before they impact downstream operations. This drastically compresses resolution windows from days to minutes, directly protecting enterprise service levels and profit margins."\n}'

In [62]:
solution = Solution.model_validate_json(interaction3.output_text)
print(solution.solution)
print(solution.description)

Autonomous Logistics Exception Resolver (ALER)
An Agentic AI system that continuously monitors freight telemetry to detect transit delays, automatically calculates optimal rerouting options, and autonomously negotiates and books backup capacity with third-party carriers.


---
## 7. Final report

Print the three validated objects together.

In [ ]:
# Print the three chained results as a single, readable report
print(f"Business Area: {business_area.business_area}")
print(f"\t - {business_area.description}")
print(f"\t - {business_area.reasoning}")
print(f"\nPain Point: {pain_point.pain_point}")
print(f"\t - {pain_point.description}")
print(f"\nSolution: {solution.solution}")
print(f"\t - {solution.description}")